# Tools 자체 AI 학습 노트

Frenchfries777 Tools(블록 HTML, Fry Sheet)의 자체 AI를 학습시키는 노트예요. 위에서부터 차례로 ▶를 누르면 돼요.

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고르고 저장해요.
2. 아래 **설정** 칸에서 `MODEL` 을 고르고, Hugging Face 토큰을 넣어요 (토큰은 huggingface.co → Settings → Access Tokens → **Write** 권한으로 만들어요).
3. **런타임 → 모두 실행** 을 누르고, 끝날 때까지 이 탭을 열어 두세요. 기본은 약 20분, 똑똑함은 약 2~3시간 걸려요.

| | 기본 | 똑똑함 |
|---|---|---|
| 바탕 모델 | Qwen3 1.7B | Qwen3 4B |
| 학습 자료 | 1,000개, 1바퀴 | 2,600개, 3바퀴 |

끝나면 마지막 칸에 사이트에 연결할 모델 주소가 나와요. 그 주소를 Claude에게 알려 주면 사이트에 연결해요.

In [ ]:
#@title 설정
MODEL = "basic"  #@param ["basic", "smart"]
HF_TOKEN = ""  #@param {type:"string"}
RUN_EVAL = True  #@param {type:"boolean"}

REPO_RAW = "https://raw.githubusercontent.com/frenchfries777mc/tools/main/train/data/"
CFG = {
  "basic": dict(base="unsloth/Qwen3-1.7B", size="1.7B", data="basic.jsonl", epochs=1, r=8,  lr=2e-4),
  "smart": dict(base="unsloth/Qwen3-4B",   size="4B",   data="full.jsonl",  epochs=3, r=16, lr=1.5e-4),
}[MODEL]
MAX_LEN = 4096
print("학습할 모델:", "기본" if MODEL == "basic" else "똑똑함", CFG)
assert HF_TOKEN.startswith("hf_"), "Hugging Face 토큰(hf_로 시작)을 넣어 주세요"

## 1. 준비 (프로그램 설치)

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
import json, re, math, random, time, urllib.request, os, subprocess
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 T4 GPU로 바꿔 주세요)")
assert torch.cuda.is_available()

def load(name):
    with urllib.request.urlopen(REPO_RAW + name) as r:
        return [json.loads(l) for l in r.read().decode("utf-8").splitlines() if l.strip()]
train_rows = load(CFG["data"]); test_rows = load("test.jsonl")
from collections import Counter
print("학습 자료", len(train_rows), dict(Counter(r["task"] for r in train_rows)))
print("시험", len(test_rows), dict(Counter(r["task"] for r in test_rows)))

## 2. 바탕 모델 불러오기

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(CFG["base"], max_seq_length=MAX_LEN, load_in_4bit=True)

def prompt_text(msgs):   # what the site sends (thinking off), up to where the answer starts
    return tokenizer.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)
def full_text(msgs):
    return tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)
print(prompt_text(train_rows[0]["messages"])[-300:])

## 3. 시험 채점 (학습 전)

시험 문제 일부를 풀게 해서 점수를 재요. 학습이 끝난 뒤 같은 문제로 다시 재서 비교해요.

In [ ]:
CAT = None
def catalog():
    global CAT
    if CAT is None:
        with urllib.request.urlopen(REPO_RAW.replace("/data/", "/ref/") + "catalog.txt") as r:
            CAT = set(l.split(" ")[0] for l in r.read().decode("utf-8").splitlines() if l.strip())
    return CAT

def chrf(hyp, ref, n=4):   # character n-gram F-score (0..1), good enough to compare translations
    def grams(s, k): s = re.sub(r"\s+", " ", s.strip()); return Counter(s[i:i+k] for i in range(max(0, len(s)-k+1)))
    ps, rs = [], []
    for k in range(1, n+1):
        h, r = grams(hyp, k), grams(ref, k); m = sum((h & r).values())
        ps.append(m / max(1, sum(h.values()))); rs.append(m / max(1, sum(r.values())))
    p, r = sum(ps)/n, sum(rs)/n
    return 0 if p + r == 0 else 5*p*r/(4*p + r)

def jload(s):
    s = s.strip(); s = re.sub(r"^```(json)?|```$", "", s).strip()
    try: return json.loads(s[s.index("{"): s.rindex("}")+1])
    except Exception: return None

def score(task, out, ref):
    if task in ("code", "und"):
        m = re.match(r"\s*PLAN:\s*(edit|new)", out); rp = re.match(r"PLAN:\s*(edit|new)", ref).group(1)
        a, b = out.find("<xml"), out.rfind("</xml>")
        if a < 0 or b < a: return 0.0
        xml = out[a:b+6]
        types = set(re.findall(r'<(?:block|shadow) type="([^"]+)"', xml))
        if not types or not types <= catalog() or xml.count('type="page_root"') != 1: return 0.25
        names = set(re.findall(r'<field name="NAME">([^<]*)</field>', xml))
        if any(n not in names for n in re.findall(r'<field name="EL">([^<]*)</field>', xml)): return 0.5
        return 1.0 if (m and m.group(1) == rp) else 0.75
    if task == "tr": return chrf(out, ref)
    j, r = jload(out), json.loads(ref)
    if not j: return 0.0
    if task == "formula":
        f = lambda s: re.sub(r"\s+", "", str(s or "")).upper()
        return 1.0 if f(j.get("formula")) == f(r["formula"]) else 0.5 if str(j.get("formula", "")).startswith("=") else 0.25
    if task == "table":
        rows = j.get("rows")
        if not isinstance(rows, list) or not rows or not all(isinstance(x, list) for x in rows): return 0.25
        w = len(rows[0]); ok = all(len(x) == w for x in rows)
        return (0.5 if ok else 0.25) + (0.25 if w == len(r["rows"][0]) else 0) + (0.25 if len(rows) == len(r["rows"]) else 0)
    if task == "graph":
        return 0.4*(j.get("type") == r["type"]) + 0.3*(j.get("series") == r["series"]) + 0.3*(j.get("range") == r["range"])
    return 0.0

random.seed(7)
EVAL = [r for t in ["code", "und", "tr", "graph", "formula", "table"] for r in random.sample([x for x in test_rows if x["task"] == t], 4)]

def run_eval(label):
    FastLanguageModel.for_inference(model)
    per = {}
    for i, r in enumerate(EVAL):
        msgs = r["messages"]; ids = tokenizer(prompt_text(msgs), return_tensors="pt").to("cuda")
        long = r["task"] in ("code", "und")
        out = model.generate(**ids, max_new_tokens=1800 if long else 300, do_sample=False, temperature=None, top_p=None, top_k=None)
        text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
        per.setdefault(r["task"], []).append(score(r["task"], text, msgs[2]["content"]))
        print(f"\r{label}: {i+1}/{len(EVAL)}", end="")
    res = {t: round(100*sum(v)/len(v)) for t, v in per.items()}
    print("\n" + label, res, "평균", round(sum(res.values())/len(res)))
    return res

before = run_eval("학습 전") if RUN_EVAL else None

## 4. 학습

In [ ]:
model = FastLanguageModel.get_peft_model(model, r=CFG["r"], lora_alpha=CFG["r"]*2, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

from datasets import Dataset
texts = [full_text(r["messages"]) for r in train_rows]
keep = [t for t in texts if len(tokenizer(t)["input_ids"]) <= MAX_LEN]
print(f"학습 자료 {len(keep)}개 (너무 긴 {len(texts)-len(keep)}개 제외)")
ds = Dataset.from_dict({"text": keep})

from trl import SFTTrainer, SFTConfig
args = dict(dataset_text_field="text", per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=CFG["epochs"], learning_rate=CFG["lr"], warmup_ratio=0.03, lr_scheduler_type="cosine",
    logging_steps=10, optim="adamw_8bit", weight_decay=0.01, seed=3407, output_dir="out", report_to="none",
    save_strategy="no", fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported())
try: cfg = SFTConfig(max_length=MAX_LEN, **args)
except TypeError: cfg = SFTConfig(max_seq_length=MAX_LEN, **args)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=cfg)

from unsloth.chat_templates import train_on_responses_only   # learn only the answers, not the prompts
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
t0 = time.time(); trainer.train(); print(f"학습 끝: {(time.time()-t0)/60:.0f}분")

## 5. 시험 채점 (학습 후)

In [ ]:
after = run_eval("학습 후") if RUN_EVAL else None
if before and after:
    names = {"code": "블록 코딩", "und": "요청 이해", "tr": "번역", "graph": "그래프", "formula": "수식", "table": "표"}
    for t in names: print(f"{names[t]:6s} {before.get(t,0):3d} → {after.get(t,0):3d}")

## 6. 브라우저용으로 바꾸고 Hugging Face에 올리기

사이트는 브라우저 안에서 도는 WebLLM 형식(MLC)을 써요. 그래픽 카드에 따라 두 가지(q4f16_1, q4f32_1)를 만들어 올려요.

In [ ]:
import os, subprocess
if not os.path.exists("merged/config.json"):
    model.save_pretrained_merged("merged", tokenizer, save_method="merged_16bit")
# The MLC converter has no wheels for Colab's newer Python, so it gets its own Python 3.11.
PY = "/content/mlcenv/bin/python"
if not os.path.exists(PY):
    sh = lambda c: subprocess.run(c, shell=True, check=True)
    sh("pip install -q uv")
    sh("uv venv -q -p 3.11 /content/mlcenv")
    sh(f"uv pip install -q --python {PY} --prerelease allow -f https://mlc.ai/wheels mlc-llm-nightly-cpu mlc-ai-nightly-cpu safetensors")
    sh(f"uv pip install -q --python {PY} --index-url https://download.pytorch.org/whl/cpu torch")
    sh(f"uv pip install -q --python {PY} psutil fastapi uvicorn shortuuid requests tqdm pydantic prompt_toolkit tiktoken transformers sentencepiece ml_dtypes openai")
MLC = [PY, "-m", "mlc_llm"]
def run(args):   # show the converter's own error instead of only its exit code
    r = subprocess.run(args, capture_output=True, text=True)
    if r.returncode:
        print((r.stdout + r.stderr)[-3000:]); raise SystemExit("위 오류를 Claude에게 보내 주세요")
# Newer transformers writes rope_theta inside rope_parameters; MLC still reads it from the top level.
import json
c = json.load(open("merged/config.json"))
rp = c.get("rope_parameters") or c.get("rope_scaling") or {}
c["rope_theta"] = c.get("rope_theta") or rp.get("rope_theta") or 1000000
if isinstance(c.get("rope_scaling"), dict) and c["rope_scaling"].get("rope_type") == "default":
    c["rope_scaling"] = None
json.dump(c, open("merged/config.json", "w"), indent=2)
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN); user = api.whoami()["name"]
links = []
for q in ["q4f16_1", "q4f32_1"]:
    name = f"FF-Tools-{CFG['size']}-{q}-MLC"; out = f"mlc/{name}"
    run(MLC + ["convert_weight", "merged", "--quantization", q, "-o", out])
    run(MLC + ["gen_config", "merged", "--quantization", q, "--conv-template", "qwen2",
               "--context-window-size", "8192", "--prefill-chunk-size", "1024", "-o", out])
    api.create_repo(f"{user}/{name}", exist_ok=True)
    api.upload_folder(folder_path=out, repo_id=f"{user}/{name}", commit_message="Tools fine-tune")
    links.append(f"https://huggingface.co/{user}/{name}")
print("\n올린 모델:"); [print(" ", l) for l in links]
print("\n이 주소를 Claude에게 알려 주세요.")